In [1]:
import os
import psycopg2
import numpy as np
from langchain_openai import OpenAIEmbeddings


c:\Users\USER\Desktop\Practice\LangChain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
texts = [
    "Type: Desktop, OS: Ubuntu, GPU: NVIDIA, CPU: AMD, RAM: 64GB, SSD: 2TB",
    "Type: Desktop, OS: Linux Mint, GPU: NVIDIA, CPU: AMD, RAM: 64GB, SSD: 2TB",
    "Type: Desktop, OS: Manjaro, GPU: NVIDIA, CPU: AMD, RAM: 64GB, SSD: 2TB",
    "Type: Desktop, OS: Windows, GPU: NVIDIA, CPU: AMD, RAM: 64GB, SSD: 2TB",
    "Type: Desktop, OS: Windows, GPU: NVIDIA, CPU: Intel, RAM: 32GB, SSD: 1TB",
    "Type: Desktop, OS: Fedora, GPU: AMD, CPU: AMD, CPU: Intel, RAM: 16GB, SSD: 1TB",
    "Type: Desktop, OS: Windows, GPU: NVIDIA, CPU: AMD, RAM: 64GB, SSD: 2TB",
    "Type: Desktop, OS: Windows, GPU: AMD, CPU: AMD, RAM: 16GB, SSD: 1TB",
    "Type: Laptop, OS: Ubuntu, GPU: NVIDIA, CPU: Intel, RAM: 16GB, SSD: 1TB",
    "Type: Laptop, OS: Windows, GPU: NVIDIA, CPU: AMD, CPU: Intel, RAM: 16GB, SSD: 1TB",
    "Type: Laptop, OS: Windows, GPU: AMD, CPU: AMD, RAM: 16GB, SSD: 500GB",
    "Type: Laptop, OS: Mac OS, GPU: NVIDIA, CPU: AMD, RAM: 16GB, SSD: 1TB",
]

In [ ]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

embeddings_list = []

for text in texts:
    embeddings_list.append(embeddings.embed_query(text))

In [6]:
len(embeddings_list[0])

1536

In [8]:
conn = psycopg2.connect("dbname=vectortutorialdb user=postgres password=mygoodlord host=localhost")
cur = conn.cursor()

for i in range(len(embeddings_list)):
    embedding = embeddings_list[i]
    content = texts[i]
    cur.execute("INSERT INTO items (content, embedding) VALUES (%s, %s)", (content, embedding))

conn.commit()

cur.close()
conn.close()

In [9]:
new_text = "Type: Desktop, OS: Arch Linux, GPU: NVIDIA, CPU: AMD, RAM: 64GB, SSD: 2TB"
new_embedding = embeddings.embed_query(new_text)

In [10]:
conn = psycopg2.connect("dbname=vectortutorialdb user=postgres password=mygoodlord host=localhost")
cur = conn.cursor()

cur.execute("""SELECT id, content
FROM items
ORDER BY embedding <-> %s::vector
LIMIT 5;
""", (new_embedding,))

In [11]:
results = cur.fetchall()
for row in results:
    print(row)

(1, 'Type: Desktop, OS: Ubuntu, GPU: NVIDIA, CPU: AMD, RAM: 64GB, SSD: 2TB')
(3, 'Type: Desktop, OS: Manjaro, GPU: NVIDIA, CPU: AMD, RAM: 64GB, SSD: 2TB')
(2, 'Type: Desktop, OS: Linux Mint, GPU: NVIDIA, CPU: AMD, RAM: 64GB, SSD: 2TB')
(4, 'Type: Desktop, OS: Windows, GPU: NVIDIA, CPU: AMD, RAM: 64GB, SSD: 2TB')
(7, 'Type: Desktop, OS: Windows, GPU: NVIDIA, CPU: AMD, RAM: 64GB, SSD: 2TB')
